# <font color="brown">Explaining a Black-Box Loan Default Model </font>

## <font color = "brown">Problem Statement </font>

### <font color="blue"> Context

Two earlier case studies (Feature Engineering, PyCaret) both landed on the same model for predicting loan default: a Gradient Boosting Classifier, verified to reach 0.911 AUC on held-out data. It works. But a bank cannot simply deploy it and move on, lending regulations in most jurisdictions require that a rejected applicant be told *why*, and a model whose reasoning nobody can articulate is a real legal and business liability, regardless of how accurate it is.

### <font color="blue"> Objective

- Explain the model **globally**: across all applicants, which features drive predictions, and in which direction, using more than one independent method.
- Explain the model **locally**: for one specific applicant, why did the model predict what it predicted, again checked with more than one method.
- Understand what each explanation technique actually assumes, and where it can mislead if used without checking those assumptions.

### <font color="blue"> Data Dictionary

The same loan application data and Gradient Boosting model from the Feature Engineering and PyCaret case studies (Age, Annual_Income, Loan_Amount, Credit_Score, Employment_Years, Num_Dependents, Existing_Loans, Home_Ownership, Purpose, target Default), rebuilt here with the identical split and random state so results match exactly.

## <font color="brown"> Importing Necessary Libraries

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.inspection import permutation_importance, PartialDependenceDisplay, partial_dependence
from sklearn.metrics import roc_auc_score

import shap
import lime.lime_tabular
from PyALE import ale

%matplotlib inline

## <font color="brown"> Rebuilding the Model (Same as the Feature Engineering / PyCaret Case Studies)

In [ ]:
df = pd.read_csv(r"../Feature Engineering/loan_applications.csv").drop(columns=["Applicant_ID"])

cat_cols = ["Home_Ownership", "Purpose"]
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df.drop(columns=["Default"]), df["Default"], test_size=0.2, random_state=1, stratify=df["Default"]
)
X_train = pd.get_dummies(X_train_raw, columns=cat_cols, drop_first=True, dtype=int)
X_test = pd.get_dummies(X_test_raw, columns=cat_cols, drop_first=True, dtype=int)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

In [ ]:
gb = GradientBoostingClassifier(random_state=1)
gb.fit(X_train, y_train)

test_proba = gb.predict_proba(X_test)[:, 1]
print("Test AUC:", round(roc_auc_score(y_test, test_proba), 4))

**Matches the 0.911 AUC from the earlier case studies**, confirming this is the same model, faithfully rebuilt, not a new one. Everything below explains *this* model, exactly as it was validated before.

## <font color="brown"> Part 1: Global Explanations, Which Features Matter Overall?

### <font color="blue"> Built-In (Impurity-Based) Feature Importance

Tree ensembles track, internally, how much each feature reduced impurity across every split in every tree, this comes for free from the fitted model, no extra computation needed:

In [ ]:
built_in_importance = pd.Series(gb.feature_importances_, index=X_train.columns)
built_in_importance.sort_values(ascending=False).head(8)

**Loan_Amount, Annual_Income, and Credit_Score dominate**, everything else contributes under 1% each. Worth knowing before trusting this number blindly: impurity-based importance can be biased toward continuous or high-cardinality features (they offer more possible split points), so it's good practice to cross-check it against a method that doesn't share that bias.

### <font color="blue"> Permutation Importance: An Independent Cross-Check

Permutation importance takes a completely different approach: shuffle one feature's values (breaking its relationship with the target) and measure how much the model's performance drops. A feature the model truly relies on will hurt performance badly when scrambled; an unused feature won't move the needle. This has no impurity-based bias, and, crucially, it's computed on the held-out **test** set, not the training data the model already memorized from.

In [ ]:
perm_result = permutation_importance(
    gb, X_test, y_test, n_repeats=20, random_state=1, scoring="roc_auc"
)
perm_importance = pd.Series(perm_result.importances_mean, index=X_test.columns)
perm_importance.sort_values(ascending=False).head(8)

**Same ranking, Loan_Amount, Annual_Income, Credit_Score, in the same order**, from a method with entirely different mechanics and computed on entirely different data (test set vs. training-set impurity). Two independent methods agreeing is real evidence this ranking reflects the model's genuine behavior, not an artifact of either method.

### <font color="blue"> Partial Dependence: Not Just *Which* Features, But *How*

Importance rankings say *which* features matter, not *how* they move the prediction. A Partial Dependence Plot (PDP) answers that: sweep one feature across its range, holding everything else at its real, observed values, and average the model's predicted probability at each point.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
PartialDependenceDisplay.from_estimator(
    gb, X_train, ["Credit_Score"], kind="average", response_method="predict_proba", method="brute", ax=ax
)
ax.set_title("Partial Dependence: Credit Score → Predicted Default Probability")
plt.show()

Predicted default probability falls steadily as Credit_Score rises, from roughly 54% at the low end down to about 15% at the high end, a large, smooth, monotonic effect exactly matching what a loan officer would expect, and a useful, concrete sanity check that the model learned something sensible.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
PartialDependenceDisplay.from_estimator(
    gb, X_train, ["Loan_Amount"], kind="average", response_method="predict_proba", method="brute", ax=ax
)
ax.set_title("Partial Dependence: Loan Amount → Predicted Default Probability")
plt.show()

Loan_Amount shows the same kind of clear, rising relationship, from about 5% up past 60% at the largest loan sizes.

### <font color="blue"> ICE Curves: Does the Average Hide Different Stories?

A PDP shows the *average* effect across all applicants. That can hide real differences, some individual applicants might respond very differently to Credit_Score than others. Individual Conditional Expectation (ICE) curves plot that same sweep separately for a sample of individual applicants, with the PDP average overlaid in bold:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
PartialDependenceDisplay.from_estimator(
    gb, X_train, ["Credit_Score"], kind="both", response_method="predict_proba", method="brute",
    subsample=60, random_state=1, ax=ax,
)
ax.set_title("ICE Curves + PDP Average: Credit Score")
plt.show()

The individual (thin) curves largely run parallel to the bold PDP average, applicants differ in their overall risk level (some curves sit higher, some lower), but nearly everyone's risk responds to Credit_Score in the same declining direction and at a similar rate. No hidden subgroup reacts the opposite way. The PDP average is a fair summary here, not masking conflicting behavior.

### <font color="blue"> SHAP: A Unified View of Direction and Magnitude, for Every Applicant

SHAP (SHapley Additive exPlanations) assigns each feature, for each individual prediction, a signed contribution: how much that feature pushed *this specific* prediction above or below the average. Averaging the size of those contributions across all applicants gives a global importance ranking that also carries direction:

In [ ]:
explainer = shap.TreeExplainer(gb)
shap_values = explainer(X_test)

shap.summary_plot(shap_values, X_test, show=False)
plt.tight_layout()
plt.show()

Reading this plot: each dot is one applicant, color shows that applicant's actual feature value (red = high, blue = low), and horizontal position shows how much that feature pushed *this applicant's* prediction toward default (right) or away from it (left). For Loan_Amount, red dots (high loan amounts) cluster to the right, confirming what the PDP already showed, but now for every individual applicant simultaneously, not just the average.

## <font color="brown"> Part 2: Explaining One Specific Decision

Global explanations describe the model in general. A rejected applicant needs something more specific: why was *my* application flagged? Take the test applicant the model is most confident will default:

In [ ]:
highest_risk_idx = np.argmax(test_proba)
applicant = X_test.iloc[highest_risk_idx]

print("Predicted default probability:", round(test_proba[highest_risk_idx], 4))
print("Actual outcome:", y_test.iloc[highest_risk_idx])
print()
print(applicant)

A 31-year-old, $14,000 annual income, requesting a $38,757 loan, credit score 517, predicted 99.8% likely to default, and this applicant did in fact default. Why did the model reach that conclusion?

### <font color="blue"> LIME: Explain This One Prediction With a Simple Local Model

LIME works by generating many small perturbations around this one applicant, seeing how the model's prediction changes, and fitting a simple, interpretable (linear) model to that local neighborhood only. It doesn't need to understand the real model globally, just well enough right around this one point.

In [ ]:
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train.values, feature_names=X_train.columns.tolist(),
    class_names=["No Default", "Default"], discretize_continuous=True, random_state=1,
)
lime_exp = lime_explainer.explain_instance(
    applicant.values, gb.predict_proba, num_features=6
)
for feature, weight in lime_exp.as_list():
    print(f"{feature:35s} {weight:+.4f}")

LIME's top three drivers, in order: Loan_Amount above $22,574, Annual_Income below $41,953, and Credit_Score below 589, each pushing the prediction toward default.

### <font color="blue"> SHAP: Explain the Same Prediction, a Different Way

In [ ]:
applicant_shap = pd.Series(shap_values.values[highest_risk_idx], index=X_test.columns)
print("Base value (average log-odds):", round(shap_values.base_values[highest_risk_idx], 4))
applicant_shap.sort_values(key=abs, ascending=False).head(6)

In [ ]:
shap.plots.waterfall(shap_values[highest_risk_idx], show=False)
plt.tight_layout()
plt.show()

**SHAP's top three drivers are the identical three features LIME found, in the identical order**: Loan_Amount, Annual_Income, Credit_Score, all pushing toward default. Two explanation methods built on completely different mathematical foundations (LIME's local linear approximation vs. SHAP's game-theoretic contribution split) independently agree on why this specific applicant was flagged, real, cross-verified confidence in the explanation, not just one method's opinion.

In [ ]:
reconstructed = shap_values.base_values[highest_risk_idx] + shap_values.values[highest_risk_idx].sum()
print("Base value + sum of all SHAP contributions:", round(reconstructed, 4))
print("Sigmoid of that (reconstructed probability):", round(1 / (1 + np.exp(-reconstructed)), 4))
print("Model's actual predicted probability:       ", round(test_proba[highest_risk_idx], 4))

SHAP values aren't just a plausible-looking ranking, they **exactly** reconstruct the model's actual predicted probability when summed, a mathematical guarantee (not true of every explanation method), and a concrete way to verify the explanation is complete and internally consistent.

## <font color="brown"> Part 3: A Caveat, Checking PDP Against ALE

PDP has a known blind spot: it averages the model's output while holding other features at their *real, observed* values, but when features are correlated, sweeping one feature can create combinations that never actually occur in reality (e.g. a very high loan amount paired with a very low income that happens to co-occur in the data purely by chance), and PDP has no way to know the difference. Accumulated Local Effects (ALE) fixes this by only averaging over *realistic*, local neighborhoods of the data, making it more robust when features are correlated. First, worth checking whether this is even a concern here:

In [ ]:
print("Correlation between Loan_Amount and Annual_Income:")
print(round(X_train[["Loan_Amount", "Annual_Income"]].corr().iloc[0, 1], 4))

Essentially zero, these two features were generated independently, so PDP's independence assumption is not violated here, and PDP should be trustworthy for this specific model. Still worth confirming directly with ALE, since "should be fine" is exactly the kind of assumption this whole case-study series has insisted on checking rather than trusting:

In [ ]:
class ProbaWrapper:
    def __init__(self, model):
        self.model = model

    def predict(self, X):
        return self.model.predict_proba(X)[:, 1]

ale_effect = ale(
    X=X_train, model=ProbaWrapper(gb), feature=["Loan_Amount"], grid_size=15, include_CI=False
)

**The ALE curve rises steadily across the Loan_Amount range, the same direction and shape as the PDP curve above.** With uncorrelated features, PDP and ALE have no reason to disagree, and they don't, a reassuring, verified confirmation. This is exactly the situation where it's safe to trust a PDP. In a dataset with genuinely correlated features, this same check could just as easily have revealed a disagreement worth investigating, which is the entire point of running it.

## <font color="brown"> Business Insights and Recommendations

- **The model's top three drivers, Loan_Amount, Annual_Income, and Credit_Score, are confirmed by two independent global methods** (built-in importance and permutation importance), and their individual effects are large, smooth, and directionally exactly what a lending officer would expect (PDP). This is a model whose behavior can be defended to a regulator or auditor.

- **Individual rejection reasons can be generated automatically and consistently**: LIME and SHAP independently agreed on the same top three reasons for the highest-risk applicant in this dataset, this is the basis for an automated, defensible adverse-action notice, not a black-box guess.

- **SHAP's guarantee that contributions sum exactly to the predicted probability** makes it the stronger choice specifically for regulatory or compliance-facing explanations, where completeness and consistency matter as much as plausibility.

- **PDP was verified, not assumed, to be trustworthy here**, its independence assumption was checked directly (near-zero correlation between the two features most likely to violate it) and cross-confirmed with ALE. This check should be repeated on any new dataset before leaning on PDP, correlated features are common in real lending data (e.g. loan amount and property value) even though they didn't appear here.

- **No single explanation method should be trusted alone.** Every method used in this notebook was cross-checked against at least one independent alternative (importance vs. permutation, LIME vs. SHAP, PDP vs. ALE), the same verification discipline used throughout this entire case-study series, applied here to the explanations themselves, not just the model's predictions.